# 1. Base Setup

## 1.1 Install packages

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost
!pip install lightgbm

## 1.2 Load all needed imports

In [18]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt
import datetime

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix, accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV
from sklearn.base import clone

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [19]:
import cf_copilot
print(cf_copilot.__file__)

/home/jeroenewalts/code/EwaltsJ/cf_copilot/cf_copilot/__init__.py


In [20]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [21]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)

# Example: train+valid for search, final holdout for test
valid_cutoff = big_df["reference_date"].quantile(0.6)
test_cutoff = big_df["reference_date"].quantile(0.8)

search_df = big_df[big_df["reference_date"] <= test_cutoff].copy()
final_test_df = big_df[big_df["reference_date"] > test_cutoff].copy()

test_fold = np.where(search_df["reference_date"] <= valid_cutoff, -1, 0)
ps = PredefinedSplit(test_fold)

X_search, y_search = preprocess(search_df)
X_final_test, y_final_test = preprocess(final_test_df)

Loading local file: /home/jeroenewalts/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [22]:
X_search.shape, y_search.shape, X_final_test.shape, y_final_test.shape, test_fold.shape

((79725, 18), (79725,), (18444, 18), (18444,), (79725,))

In [23]:
print("Train rows:", np.sum(test_fold == -1))
print("Validation rows:", np.sum(test_fold == 0))
print("Test rows:", len(X_final_test))

Train rows: 60018
Validation rows: 19707
Test rows: 18444


In [24]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [25]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        num_class=7,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=4,
        tree_method="hist",
        verbosity=0,
        colsample_bytree=0.7265477506155757,
        gamma=0.058794858725743554,
        learning_rate=0.024522728891053808,
        max_depth=7,
        min_child_weight=2,
        n_estimators=352,
        reg_alpha=0.5867511656638482,
        reg_lambda=1.930510614528276,
        subsample=0.8821102743060054,
    ))
])

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(
        objective="multiclass",
        random_state=42,
        n_jobs=4,
        verbose=-1,
        colsample_bytree=0.7992694074557947,
        learning_rate=0.026355835028602365,
        max_depth=8,
        min_child_samples=43,
        min_child_weight=0.13418447132349934,
        min_split_gain=0.11825955754154543,
        n_estimators=846,
        num_leaves=15,
        reg_alpha=0.38292687475378984,
        reg_lambda=0.9717120953891037,
        subsample=0.954674147279825,
        subsample_freq=1,
    ))
])

histgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        loss="log_loss",
        random_state=42,
        early_stopping=False,
        verbose=0,
        l2_regularization=0.15691276268453191,
        learning_rate=0.013549104078164053,
        max_bins=64,
        max_depth=12,
        max_features=0.9343920482048823,
        max_iter=592,
        max_leaf_nodes=26,
        min_samples_leaf=10,
    ))
])

In [26]:
# XGBoost-compatible labels for the whole ensemble
y_search_ens = y_search - 1
y_final_test_ens = y_final_test - 1

print("START:", datetime.datetime.now())

ensemble_classifier = VotingClassifier(
    estimators=[
        ("xgb", xgb_pipeline),
        ("lgbm", lgbm_pipeline),
        ("hist", histgb_pipeline),
    ],
    voting="soft",
    weights=[2, 1, 1],
    n_jobs=1,
)

ensemble_model = ensemble_classifier
ensemble_model.fit(X_search, y_search_ens)

print("END FIT:", datetime.datetime.now())

START: 2026-03-25 11:21:04.583832
END FIT: 2026-03-25 11:21:31.241299


In [27]:
y_proba = ensemble_model.predict_proba(X_final_test)
y_pred = ensemble_model.predict(X_final_test)

clf_classes = np.array(ensemble_model.classes_)

search_classes = np.array(sorted(pd.Series(y_search_ens).unique()))
test_classes = np.array(sorted(pd.Series(y_final_test_ens).unique()))

missing_in_train = set(test_classes) - set(clf_classes)
print("Classes in final test but not seen during search:", missing_in_train)

if missing_in_train:
    raise ValueError(
        f"Final test contains unseen classes: {missing_in_train}. "
        "The model cannot evaluate those correctly."
    )

if y_proba.shape[1] != len(clf_classes):
    raise ValueError(
        f"Mismatch between probability columns ({y_proba.shape[1]}) "
        f"and classifier classes ({len(clf_classes)})."
    )

final_log_loss = log_loss(y_final_test_ens, y_proba, labels=clf_classes)
final_accuracy = accuracy_score(y_final_test_ens, y_pred)
final_f1_macro = f1_score(y_final_test_ens, y_pred, average="macro")

print("\nEnsemble holdout performance")
print(f"log_loss  : {final_log_loss:.6f}")
print(f"accuracy  : {final_accuracy:.6f}")
print(f"f1_macro  : {final_f1_macro:.6f}")

print("\nClass checks")
print("Search classes:", list(search_classes))
print("Test classes  :", list(test_classes))
print("Model classes :", list(clf_classes))

/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Classes in final test but not seen during search: set()

Ensemble holdout performance
log_loss  : 0.698734
accuracy  : 0.756940
f1_macro  : 0.568688

Class checks
Search classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Test classes  : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Model classes : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [28]:
from cf_copilot.cashflow_prediction.evaluation import build_actual_weekly_cf, compare_forecast_vs_actual, compute_forecast_metrics
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES
from colorama import Fore, Style
from cf_copilot.ml_logic.encoders import preprocess


def evaluate_forecast_holdout(
    model: Pipeline,
    test_df: pd.DataFrame,
    verbose: bool = True
) -> tuple[dict, dict]:
    """
    Evaluate forecast quality on the holdout set by reference_date.

    Args:
        model: fitted sklearn Pipeline
        test_df: holdout snapshot dataframe
        verbose: whether to print evaluation output

    Returns:
        forecast_metrics: scalar metrics to merge into MLflow metrics
        forecast_summary: JSON-serializable artifact with per-reference-date detail
    """
    if verbose:
        print(Fore.BLUE + f"\nEvaluating forecast on {len(test_df)} rows..." + Style.RESET_ALL)

    if model is None:
        if verbose:
            print("❌ No model to evaluate forecast")
        return {}, {}

    if len(test_df) == 0:
        if verbose:
            print("❌ No holdout rows available for forecast evaluation")
        return {}, {}

    per_reference_results = []
    if hasattr(model, "named_steps") and "classifier" in model.named_steps:
        model_classes = np.array(model.named_steps["classifier"].classes_)
    else:
        model_classes = np.array(model.classes_)

    # Support both label schemes:
    # standard models: 1..7
    # XGBoost: 0..6
    if np.array_equal(np.sort(model_classes), np.array([0, 1, 2, 3, 4, 5, 6])):
        class_to_index = {int(c) + 1: i for i, c in enumerate(model_classes)}
    else:
        class_to_index = {int(c): i for i, c in enumerate(model_classes)}

    for reference_date, snapshot_df in test_df.groupby("reference_date"):
        snapshot_df = snapshot_df.copy()
        if len(snapshot_df) == 0:
            continue

        X_snapshot, _ = preprocess(snapshot_df)
        probas = model.predict_proba(X_snapshot)

        pred_cash_df = snapshot_df.copy()

        for w in WEEK_CLASSES:
            pred_cash_df[f"p_{w}"] = (
                probas[:, class_to_index[int(w)]]
                if int(w) in class_to_index
                else 0.0
            )

        weekly_forecast_df = pd.DataFrame([
            {
                "week_bucket": int(w),
                "forecast_cash": round(
                    float((pred_cash_df["total_open_amount"] * pred_cash_df[f"p_{w}"]).sum()),
                    2,
                ),
            }
            for w in WEEK_CLASSES
        ])

        total_invoice_amount = round(float(pred_cash_df["total_open_amount"].sum()), 2)
        total_expected_cash = round(float(weekly_forecast_df["forecast_cash"].sum()), 2)

        if total_expected_cash - total_invoice_amount > 0:
            print("Expected cash exceeds total invoice amount. This should not happen.")

        actual_weekly_df = build_actual_weekly_cf(
            invoices_df=snapshot_df,
            reference_date=pd.Timestamp(reference_date),
        )

        comparison_df = compare_forecast_vs_actual(
            weekly_forecast_df=weekly_forecast_df,
            actual_weekly_df=actual_weekly_df,
        )

        snapshot_metrics = compute_forecast_metrics(comparison_df)

        per_reference_results.append({
            "reference_date": str(pd.Timestamp(reference_date).date()),
            "forecast_check": {
                "total_invoice_amount": total_invoice_amount,
                "total_expected_cash": total_expected_cash,
            },
            "forecast_metrics": {
                "forecast_mae_weekly": float(snapshot_metrics["MAE (weekly)"]),
                "forecast_mape_weekly": (
                    float(snapshot_metrics["MAPE (weekly)"])
                    if pd.notna(snapshot_metrics["MAPE (weekly)"])
                    else None
                ),
                "total_actual_cf": float(snapshot_metrics["Total actual cf"]),
                "total_forecast_cf": float(snapshot_metrics["Total forecast cf"]),
                "total_cf_difference": float(snapshot_metrics["Total cf difference"]),
            },
        })

    if not per_reference_results:
        if verbose:
            print("❌ No per-reference-date forecast results available")
        return {}, {}

    mae_values = [r["forecast_metrics"]["forecast_mae_weekly"] for r in per_reference_results]
    mape_values = [
        r["forecast_metrics"]["forecast_mape_weekly"]
        for r in per_reference_results
        if r["forecast_metrics"]["forecast_mape_weekly"] is not None
    ]
    total_actual_values = [r["forecast_metrics"]["total_actual_cf"] for r in per_reference_results]
    total_forecast_values = [r["forecast_metrics"]["total_forecast_cf"] for r in per_reference_results]
    total_diff_values = [r["forecast_metrics"]["total_cf_difference"] for r in per_reference_results]

    forecast_metrics = {
        "forecast_mae_weekly": float(np.mean(mae_values)),
        "forecast_mape_weekly": float(np.mean(mape_values)) if mape_values else np.nan,
        "forecast_total_actual_cf": float(np.mean(total_actual_values)),
        "forecast_total_forecast_cf": float(np.mean(total_forecast_values)),
        "forecast_total_cf_difference": float(np.mean(total_diff_values)),
    }

    forecast_summary = {
        "per_reference_date": per_reference_results,
        "aggregate": {
            "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
            "forecast_mape_weekly": (
                forecast_metrics["forecast_mape_weekly"]
                if pd.notna(forecast_metrics["forecast_mape_weekly"])
                else None
            ),
            "forecast_total_actual_cf": forecast_metrics["forecast_total_actual_cf"],
            "forecast_total_forecast_cf": forecast_metrics["forecast_total_forecast_cf"],
            "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"]
        },
    }

    if verbose:
        print(f"✅ Forecast MAE weekly: {forecast_metrics['forecast_mae_weekly']:.2f}")

        if pd.notna(forecast_metrics["forecast_mape_weekly"]):
            print(f"✅ Forecast MAPE weekly: {forecast_metrics['forecast_mape_weekly']:.4f}")
        else:
            print("✅ Forecast MAPE weekly: None")

        print(f"✅ Forecast total actual cf: {forecast_metrics['forecast_total_actual_cf']:.2f}")
        print(f"✅ Forecast total forecast cf: {forecast_metrics['forecast_total_forecast_cf']:.2f}")
        print(f"✅ Forecast total cf difference: {forecast_metrics['forecast_total_cf_difference']:.2f}")

    return forecast_metrics, forecast_summary

In [29]:
forecast_metrics, forecast_summary = evaluate_forecast_holdout(
    model=ensemble_model,
    test_df=final_test_df,
    verbose=True,
)

print("\nAggregate forecast metrics")
for k, v in forecast_metrics.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics])


Evaluating forecast on 18444 rows...


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.

✅ Forecast MAE weekly: 538965.64
✅ Forecast MAPE weekly: 0.3345
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27830114.80
✅ Forecast total cf difference: -76003.33

Aggregate forecast metrics
forecast_mae_weekly: 538965.6425
forecast_mape_weekly: 0.3345
forecast_total_actual_cf: 27906118.129499994
forecast_total_forecast_cf: 27830114.800499998
forecast_total_cf_difference: -76003.329


,forecast_mae_weekly,forecast_mape_weekly,forecast_total_actual_cf,forecast_total_forecast_cf,forecast_total_cf_difference
0,538965.6425,0.3345,2.790612e+07,2.783011e+07,-76003.329


In [30]:
comparison_df = pd.DataFrame([
    {
        "model": "XGBoost",
        "log_loss": 0.703987,
        "accuracy": 0.756940,
        "f1_macro": 0.568073,
        "forecast_mae_weekly": 535341.07,
        "forecast_mape_weekly": 0.325,
        "forecast_total_cf_difference": -81844.183,
    },
    {
        "model": "VotingEnsemble_XGB_LGBM_HistGB",
        "log_loss": final_log_loss,
        "accuracy": final_accuracy,
        "f1_macro": final_f1_macro,
        "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"],
    },
])

comparison_df

,model,log_loss,accuracy,f1_macro,forecast_mae_weekly,forecast_mape_weekly,forecast_total_cf_difference
0,XGBoost,0.703987,0.75694,0.568073,535341.0700,0.3250,-81844.183
1,VotingEnsemble_XGB_LGBM_HistGB,0.698734,0.75694,0.568688,538965.6425,0.3345,-76003.329


In [31]:
weight_grid = [
    (1, 1, 1),
    (2, 1, 1),
    (3, 1, 1),
    (4, 1, 1),
    (5, 1, 1),
    (3, 2, 1),
    (3, 1, 2),
    (2, 1, 0),  # XGB + LGBM
    (2, 0, 1),  # XGB + HistGB
]

results = []

print("START:", datetime.datetime.now())

for wx, wl, wh in weight_grid:
    print(f"\nTesting weights: XGB={wx}, LGBM={wl}, HistGB={wh}")

    estimators = [("xgb", xgb_pipeline)]
    weights = [wx]

    if wl > 0:
        estimators.append(("lgbm", lgbm_pipeline))
        weights.append(wl)

    if wh > 0:
        estimators.append(("hist", histgb_pipeline))
        weights.append(wh)

    ensemble_model = VotingClassifier(
        estimators=estimators,
        voting="soft",
        weights=weights,
        n_jobs=1,
    )

    ensemble_model.fit(X_search, y_search_ens)

    # ML evaluation
    y_proba = ensemble_model.predict_proba(X_final_test)
    y_pred = ensemble_model.predict(X_final_test)

    clf_classes = np.array(ensemble_model.classes_)

    final_log_loss = log_loss(y_final_test_ens, y_proba, labels=clf_classes)
    final_accuracy = accuracy_score(y_final_test_ens, y_pred)
    final_f1_macro = f1_score(y_final_test_ens, y_pred, average="macro")

    # Forecast evaluation
    forecast_metrics, _ = evaluate_forecast_holdout(
        model=ensemble_model,
        test_df=final_test_df,
        verbose=False,
    )

    results.append({
        "weights": (wx, wl, wh),
        "log_loss": final_log_loss,
        "accuracy": final_accuracy,
        "f1_macro": final_f1_macro,
        "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"],
    })

print("END:", datetime.datetime.now())

ensemble_results_df = pd.DataFrame(results)

ensemble_results_df = ensemble_results_df.sort_values(
    by=["forecast_mae_weekly", "forecast_mape_weekly"]
).reset_index(drop=True)

ensemble_results_df

START: 2026-03-25 11:21:37.964433

Testing weights: XGB=1, LGBM=1, HistGB=1


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: 


Testing weights: XGB=2, LGBM=1, HistGB=1


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: 


Testing weights: XGB=3, LGBM=1, HistGB=1


...

/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: 


Testing weights: XGB=2, LGBM=0, HistGB=1


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


END: 2026-03-25 11:26:13.857928


,weights,log_loss,accuracy,f1_macro,forecast_mae_weekly,forecast_mape_weekly,forecast_total_cf_difference
0,"(2, 0, 1)",0.697711,0.757374,0.568796,535129.3605,0.3325,-77274.8295
1,"(5, 1, 1)",0.699488,0.757536,0.569060,535148.3055,0.3280,-78506.5550
2,"(4, 1, 1)",0.699146,0.757645,0.569375,535862.7695,0.3290,-77950.2820
3,"(3, 1, 1)",0.698838,0.757319,0.568784,536863.0175,0.3310,-77171.5015
4,"(2, 1, 0)",0.703225,0.756723,0.570357,537577.0780,0.3280,-78625.7310
5,"(2, 1, 1)",0.698734,0.756940,0.568688,538965.6425,0.3345,-76003.3290
6,"(3, 2, 1)",0.700112,0.757103,0.570401,539192.1770,0.3310,-76341.0540
7,"(3, 1, 2)",0.697913,0.757319,0.567975,539951.0980,0.3365,-75665.6065
8,"(1, 1, 1)",0.699501,0.756235,0.567038,545240.8250,0.3365,-74056.3790


In [32]:
xgb_baseline = pd.DataFrame([{
    "weights": "XGBoost only",
    "log_loss": 0.703987,
    "accuracy": 0.756940,
    "f1_macro": 0.568073,
    "forecast_mae_weekly": 535341.07,
    "forecast_mape_weekly": 0.325,
    "forecast_total_cf_difference": -81844.183,
}])

comparison_df = pd.concat([xgb_baseline, ensemble_results_df], ignore_index=True)
comparison_df

,weights,log_loss,accuracy,f1_macro,forecast_mae_weekly,forecast_mape_weekly,forecast_total_cf_difference
0,XGBoost only,0.703987,0.756940,0.568073,535341.0700,0.3250,-81844.1830
1,"(2, 0, 1)",0.697711,0.757374,0.568796,535129.3605,0.3325,-77274.8295
2,"(5, 1, 1)",0.699488,0.757536,0.569060,535148.3055,0.3280,-78506.5550
3,"(4, 1, 1)",0.699146,0.757645,0.569375,535862.7695,0.3290,-77950.2820
4,"(3, 1, 1)",0.698838,0.757319,0.568784,536863.0175,0.3310,-77171.5015
5,"(2, 1, 0)",0.703225,0.756723,0.570357,537577.0780,0.3280,-78625.7310
6,"(2, 1, 1)",0.698734,0.756940,0.568688,538965.6425,0.3345,-76003.3290
7,"(3, 2, 1)",0.700112,0.757103,0.570401,539192.1770,0.3310,-76341.0540
8,"(3, 1, 2)",0.697913,0.757319,0.567975,539951.0980,0.3365,-75665.6065
9,"(1, 1, 1)",0.699501,0.756235,0.567038,545240.8250,0.3365,-74056.3790
